In [ ]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe
from utils.kafka_producer_adapter import run_send_queue_producer_once, run_send_queue_producer_loop, build_sensor_message_payload, json_dumps_safe

### One Batch

In [ ]:


engine = get_engine_from_env()


In [ ]:

result = run_send_queue_producer_once(
    engine=engine,
    dataset_id="pump_synthetic_v1",
    run_id="premelt_run_001",
    schema="capstone",
    queue_table="synthetic_sensor_messages_send_queue",
    control_table="simulation_state_control",
    producer_worker_id="producer_worker_001",
    client_id="pump-telemetry-producer",
    flush_timeout_seconds=30.0,
)


In [ ]:

result

----

### Loop

In [ ]:
results = run_send_queue_producer_loop(
    engine=engine,
    dataset_id="pump_synthetic_v1",
    run_id="premelt_run_001",
    schema="capstone",
    queue_table="synthetic_sensor_messages_send_queue",
    control_table="simulation_state_control",
    producer_worker_id="producer_worker_001",
    client_id="pump-telemetry-producer",
    max_batches=10,
    stop_on_failure=True,
    flush_timeout_seconds=30.0,
)



In [ ]:

results

----


### Preview Payload Before Sending

In [ ]:


preview_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT *
    FROM capstone.synthetic_sensor_messages_send_queue
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 1
    """
)

payload = build_sensor_message_payload(preview_dataframe.iloc[0].to_dict())
print(json_dumps_safe(payload))